# PACE MiniGrid on one Kaggle GPU

Set **Accelerator** to GPU and enable **Internet** in the notebook settings. This notebook runs four small PACE rollout batches, with a fresh process restoring the checkpoint at batch two. Saved checkpoints and evaluation JSON go under `/kaggle/working/pace`.

In [ ]:
import os
import shutil
import subprocess
import tempfile
import zipfile
from pathlib import Path

WORK = Path('/kaggle/working')
REPO = WORK / 'PACE-Research'
REPO_URL = 'https://github.com/SimforianusJonathan/PACE-Research.git'
REPO_REF = 'main'  # Set to a commit from the Kaggle implementation for reproducibility.
SOURCE = ''  # Optional path to an uploaded source directory or Git archive ZIP.
RUNTIME = Path('/tmp/pace-kaggle')

if not REPO.exists():
    if SOURCE:
        source = Path(SOURCE)
        if source.is_dir():
            shutil.copytree(source, REPO)
        else:
            with zipfile.ZipFile(source) as archive:
                archive.extractall(REPO)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
        subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO, check=True)

if not (REPO / 'scripts/setup_kaggle.sh').is_file():
    raise RuntimeError('This source lacks Kaggle support. Use a Git ref or source archive containing this notebook.')
if (REPO / '.git').exists():
    subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True)

env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'CUDA_VISIBLE_DEVICES': '0',
       'JAX_PLATFORMS': 'cuda', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false'}
# Let the pinned wheel libraries take precedence over the base image CUDA toolkit.
env.pop('LD_LIBRARY_PATH', None)
subprocess.run(['nvidia-smi', '-L'], env=env, check=True)

In [ ]:
subprocess.run(['bash', 'scripts/setup_kaggle.sh'], cwd=REPO, env=env, check=True)
PACE_PYTHON = str(RUNTIME / 'venv/bin/python')

probe = '''
import importlib.metadata as metadata
import json
import jax
import jax.numpy as jnp
devices = jax.local_devices(backend='gpu')
assert len(devices) == 1, devices
result = jax.jit(lambda x: x @ x)(jnp.ones((16, 16))).block_until_ready()
assert result.device.platform == 'gpu' and float(result[0, 0]) == 16.0
pin = '0f8f1284677375b889e4f13a32c9617cd009f8c4'
origin = json.loads(metadata.distribution('jaxued').read_text('direct_url.json'))
assert origin['vcs_info']['commit_id'] == pin
print({'device': str(devices[0]), 'jax': jax.__version__, 'jaxued_commit': pin})
'''
subprocess.run([PACE_PYTHON, '-c', probe], cwd=REPO, env=env, check=True)

In [ ]:
runs = WORK / 'pace'
runs.mkdir(exist_ok=True)
run = Path(tempfile.mkdtemp(prefix='smoke-', dir=runs))
base = [PACE_PYTHON, '-m', 'train.pace_minigrid']
subprocess.run(base + ['smoke', '--device', 'gpu', '--output-dir', str(run),
                       '--checkpoint-interval', '2', '--stop-after', '2'],
               cwd=REPO, env=env, check=True)
subprocess.run(base + ['resume', '--device', 'gpu', '--output-dir', str(run),
                       '--checkpoint', str(run / 'checkpoints/step_00000002.msgpack'),
                       '--checkpoint-interval', '2'], cwd=REPO, env=env, check=True)
final = run / 'checkpoints/step_00000004.msgpack'
assert final.is_file(), final
print('Resumed checkpoint:', final)

In [ ]:
subprocess.run(base + ['eval', '--device', 'gpu', '--checkpoint', str(final),
                       '--suite', 'reference', '--levels', 'Maze3', '--episodes', '2',
                       '--horizon', '16', '--output-dir', str(run / 'eval')],
               cwd=REPO, env=env, check=True)